In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from astropy.table import Table, hstack

In [ ]:
import sys
sys.path.append("..")

In [ ]:
import desi_utils

In [ ]:
import arya
arya.use("mnras")

In [ ]:
import ppxf

In [ ]:
dwarf_main = Table.read("../data/dwarfs_good.fits", hdu=1)
dwarf_fsf = Table.read("../data/dwarfs_good.fits", hdu=4)

In [ ]:
dwarf_cat = hstack([dwarf_main, dwarf_fsf])

In [ ]:
CA_TRIPLET = np.array([8500.36, 8544.44, 8664.52] )

In [ ]:
MG_TRIPLET = np.array([5168.76, 5174.12, 5185.05])

In [ ]:
CA_II_HK = ppxf.ppxf_util.air_to_vac([3968.47, 3933.66])

In [ ]:
NA_I_D = ppxf.ppxf_util.air_to_vac([5895.92, 5889.95]) # Fraunhofer D1 and D2

In [ ]:
def get_xlim(lines):
    xlim = np.min(lines)-50, np.max(lines) + 50
    return xlim


In [ ]:
def plot_stellar_line(spec, lines, label):
    fig, ax = plt.subplots(figsize=(8, 2))

    spec.plot(xlim=xlim, plot_uncertainty=True, restframe=True, label="observed spectrum")
  
    y0 = np.min(spec.flux[(spec.wavelength > xlim[0] ) & (xlim[1] > spec.wavelength)])
    y0 -= 5
    plt.scatter(lines, np.full_like(lines, y0), color="C2", s=10)
    plt.annotate(label, (np.max(lines), y0), (1, 0), color="C2", xycoords="data", textcoords="offset fontsize", va="center")



In [ ]:
plt.scatter(dwarf_cat["MAG_G"] - dwarf_cat["MAG_R"], dwarf_cat["SNR_R"], s=0.5, lw=0, color="k");

In [ ]:
plt.scatter(dwarf_cat["LOG_MSTAR_M24"], dwarf_cat["SNR_R"], s=0.5, lw=0, color="k");

In [ ]:
plt.scatter(dwarf_cat["Z"], dwarf_cat["SNR_R"], s=0.5, lw=0, color="k");

In [ ]:
plt.scatter(dwarf_cat["HALPHA_EW"], dwarf_cat["SNR_R"], s=0.5, lw=0, color="k");
plt.xlim(0, 500)

In [ ]:
g_r = dwarf_cat["MAG_G"] - dwarf_cat["MAG_R"]

In [ ]:
filt_similar = g_r < 0.5
filt_similar &= g_r > 0.4
filt_similar &= dwarf_cat["LOG_MSTAR_M24"] > 8.8
filt_similar &= dwarf_cat["LOG_MSTAR_M24"] < 9.0
filt_similar &= dwarf_cat["Z"] < 0.05
filt_similar &= dwarf_cat["IS_BGS_BRIGHT"]
filt_similar &= dwarf_cat["HALPHA_EW"] < 50
filt_similar &= dwarf_cat["HALPHA_EW"] > 20

In [ ]:
dwarf_similar = dwarf_cat[filt_similar]

In [ ]:
get_target_with_snr(14)

In [ ]:
plt.hist(dwarf_cat["SNR_R"]);
plt.hist(dwarf_cat["SNR_R"][filt_similar])
plt.xlabel("SNR in R-band")
plt.ylabel("counts")
plt.yscale("log")

In [ ]:
SNRS = [2, 3, 5, 8, 10, 15, 25]

In [ ]:
def get_target_with_snr(snr):
    idx = np.argmin(np.abs(snr - dwarf_similar["SNR_B"]))
    return int(dwarf_similar["TARGETID_1"][idx])

In [ ]:
targetids = [get_target_with_snr(snr) for snr in SNRS]

In [ ]:
spectra = desi_utils.get_data.get_spectra(targetids)

In [ ]:
spectra.flux

In [ ]:
np.sum(spectra.mask == 1)

In [ ]:
spectra.uncertainty.unit**-0.5

In [ ]:
spectra.uncertainty[i]

In [ ]:
idx_sort

In [ ]:
for i in range(7):
    plt.plot(spectra.flux[i].value / spectra.uncertainty[i].array.reshape(-1)**-0.5 + 100*i)
    plt.xlim(5500, 6500)
    plt.axhline(100*i)
    print(snr_actual[i])

In [ ]:
def get_snr(spectra, i):
    
    x = spectra.spectral_axis.value
    y = spectra.flux[i].value

    y[spectra.mask[i] == 1] = np.nan

    yerr = spectra.uncertainty[i].array ** -0.5
    filt = (5500 < x) & (x < 6000)
    
    snr = np.nanmedian((y / yerr)[filt])
    return snr

In [ ]:
snr_actual = [get_snr(spectra, i) for i in range(len(SNRS))]

In [ ]:
snr_actual

In [ ]:
idx_sort = np.argsort(snr_actual)

In [ ]:
np.array(snr_actual)[idx_sort]

In [ ]:
spectra.meta["specid"]

In [ ]:
def get_snr_r(targetid):
    return dwarf_cat[dwarf_cat["TARGETID_1"] == targetid]["SNR_R"][0]

In [ ]:
snr_r = [get_snr_r(id) for id in spectra.meta["specid"]]

In [ ]:
snr_r

In [ ]:
np.array(snr_r)[idx_sort]

In [ ]:
snr_actual

In [ ]:
desi_utils.spectral_lines.format_line("He1r_5008A")

In [ ]:
def plot_lines(y0, xlim=None, lines=desi_utils.EMISSION_LINES, restframe=True):

    for line in lines:
        wave = desi_utils.get_wavelength(line)

        if xlim is not None and not (xlim[0] < wave < xlim[1]):
            continue

        plt.scatter(wave, y0, lw=0, s=0)
        plt.annotate(desi_utils.spectral_lines.format_line(line), (wave, y0), (0, 2), 
                     xycoords="data", textcoords="offset fontsize", 
                     fontsize=6, rotation=90, ha="center",
                     arrowprops=dict(arrowstyle="-", linewidth=1, color="black"),
                    )

In [ ]:
def plot_spectrum(spectra, ii, scale =1, offset=0, xlim=None, log=False, log_min=-2):
    i = idx_sort[ii]
    
    z = spectra.meta["redshift"][i]
    
    x = spectra.spectral_axis.value / (1 + z)
    y = spectra.flux[i].value

    y[spectra.mask[i] == 1] = np.nan

    if xlim is not None:
        filt = x > xlim[0]
        filt &= x < xlim[1]

        x = x[filt]
        y = y[filt]
        x0 = xlim[0]
    else:
        x0 = x[0]

    if log:
        y_plot = np.log(y / np.nanmedian(y)) 
        y_plot[y_plot < log_min] = np.nan
        y_plot = y_plot / scale + offset
    else:
        y_plot = y / np.nanmedian(y)
        y_plot = y_plot / scale + offset
    
    plt.plot(x, y_plot, color="k", lw=1)
    
    snr = snr_actual[i]
    y0 = np.nanmedian(y_plot[x < np.quantile(x, 0.05)])

    plt.annotate(f"SNR = {snr:0.0f}", (x0, y0), (1, 1), fontsize=6, va="bottom", textcoords="offset fontsize", color="C2")

    new_offset = np.nanmax(y_plot)
    return max(new_offset, offset + 1)

In [ ]:
def plot_all_spectra(spectra, SNRS, scale=10, xlim=None, log=False, **kwargs):
    fig, ax = plt.subplots(figsize=(8, 4))
    y_offset = 0
    
    for i in range(len(SNRS)):
        y_offset = plot_spectrum(spectra, i, scale = scale, xlim=xlim, offset=y_offset, log=log, **kwargs)
        if xlim is None:
            x0 = np.min(spectra.spectral_axis.value)
        else:
            x0 = xlim[0]


    if xlim is not None:
        # idx_max = int(np.argmax(SNRS))
        # spec_top = spectra[idx_max]
        # spec_top.meta["redshift"] = spec_top.meta["redshift"][idx_max]
        plt.xlim(xlim)
    plot_lines(y_offset, xlim)

    if log:
        ylabel = "log flux"
    else:
        ylabel = "flux"
        
    ax.set(
        xlabel = "restframe wavelength / $\\AA$",
        ylabel = ylabel
    )

In [ ]:
plot_all_spectra(spectra, SNRS, scale=4, log=True)

In [ ]:
plot_all_spectra(spectra, SNRS, scale=20, log=False)

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(3600, 3850),  scale=10)

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(3800, 4000), scale=2)

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(4000, 4350), scale=2)

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(4350, 4480), scale=3)

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(4480, 4800), scale=1)

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(4800, 5100), scale=10)

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(5600, 6000), scale=1.5)


In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(6000, 6500), scale=2)

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(6500, 6670), scale=10)

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(6670, 6800), scale=5)

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(6800, 7800), log=False, scale=1)

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(7800, 8800), log=False, scale=1)

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(8800, 9800), log=False, scale=5)

# Stellar lines

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(8400, 8774), scale=3)
plt.scatter(CA_TRIPLET, np.full_like(CA_TRIPLET, -0.5), marker="|", lw=1, s=20)
plt.title("Calcium triplet")

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(5068, 5285), scale=1)
plt.scatter(MG_TRIPLET, np.full_like(MG_TRIPLET, 0), marker="|", lw=1, s=20)
plt.title("Mg I B triplet")

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(5791, 5997), scale=0.5)
plt.scatter(NA_I_D, np.full_like(NA_I_D, -1), marker="|", lw=1, s=20)
plt.title("Na I D")

In [ ]:
plot_all_spectra(spectra, SNRS, xlim=(3850, 4050), scale=0.5)
plt.scatter(CA_II_HK, np.full_like(CA_II_HK, -1), marker="|", lw=1, s=20)
plt.title("Ca II H\\&K")